# 21차시: [실습] 분류 분석(Logistic Regression)으로 상승/하락 예측

## 학습 목표
- 분류 문제의 개념과 회귀와의 차이점 이해
- 로지스틱 회귀(LogisticRegression)로 주가 상승/하락 예측 모델 구축
- FRED API로 환율 데이터를 추가 특성으로 활용
- 분류 모델 평가 지표 (Accuracy, Precision, Recall, F1) 학습

## 학습 내용
1. 이진분류란?
2. 데이터 수집 (주가 + 환율)
3. 특성 공학 (Feature Engineering)
4. LogisticRegression 모델 학습
5. 모델 평가 및 시각화

## 중요 주의사항 (Warning)

본 교재와 실습에서 사용하는 모든 데이터, 모델, 기법은
오직 머신러닝·분류 개념 이해와 교육 목적을 위한 예제입니다. 따라서,
실제 금융 시장의 복잡성, 리스크, 거래 비용, 정책·심리 요인 등을 전혀 반영하지 못하며, 실제 투자 판단이나 매매 전략에 사용해서는 안 됩니다.

👉 실습 결과는 “참고용·학습용”으로만 활용하시기 바랍니다.

In [1]:
!pip install -Uq finance-datareader koreanize-matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 34.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from datetime import datetime, timedelta
from IPython.display import display

---
## 1. 이진분류란?

### 회귀 vs 분류
| 구분 | 회귀 (Regression) | 분류 (Classification) |
|------|-------------------|----------------------|
| 출력 | 연속값 (주가) | 범주 (상승/하락) |
| 질문 | 내일 주가는 얼마? | 내일 오를까 내릴까? |
| 알고리즘 | LinearRegression | LogisticRegression |
| 평가 | MSE, R2 | Accuracy, F1 |

### 이진분류 문제 정의
```
입력(X): 오늘의 주가 관련 특성들 + 환율 데이터
출력(y): 내일 상승(1) 또는 하락(0)
```

### LogisticRegression 원리
- 이름에 "Regression"이 있지만 **분류** 알고리즘
- Sigmoid 함수로 출력을 0~1 사이의 확률로 변환
- 확률이 0.5 이상이면 1(상승), 미만이면 0(하락)

$$
P(y=1 \mid X) = \frac{1}{1 + e^{-z}}
$$

$$
z = \beta_0 + \beta_1 x_1 + \cdots + \beta_n x_n
$$

---
## 2. 데이터 수집 (주가 + 환율)

### 2.1 주가 데이터 수집 (FinanceDataReader)

In [10]:
import FinanceDataReader as fdr

# 날짜 설정 (최근 1년)
end_date = datetime.now()
start_date = end_date - timedelta(days=365)

print("[주가 데이터 수집]")
print("=" * 50)
print(f"기간: {start_date.strftime('%Y-%m-%d')} ~ {end_date.strftime('%Y-%m-%d')}")
print("종목: 삼성전자 (005930)")

# 삼성전자 주가 데이터 수집 (FinanceDataReader - Module 01/02와 동일)
df = fdr.DataReader("005930", start_date, end_date)

print(f"\n수집된 데이터: {len(df)}개")
display(df.tail())

[주가 데이터 수집]
기간: 2025-01-09 ~ 2026-01-09
종목: 삼성전자 (005930)

수집된 데이터: 242개


,Open,High,Low,Close,Volume,Change
Date,,,,,,
2026-01-05,134600,138600,133600,138100,42863376,0.074708
2026-01-06,135300,139300,132700,138900,45321341,0.005793
2026-01-07,143500,144400,137600,141000,46317413,0.015119
2026-01-08,138300,144500,138300,138800,41449329,-0.015603
2026-01-09,136000,140700,135200,139000,29514279,0.001441


### 2.2 환율 데이터 수집 (FRED API)

FRED에서 USD/KRW 환율 데이터를 가져옵니다. (Module 02에서 학습한 내용 활용)

In [12]:
print("[환율 데이터 수집 - FRED]")
print("=" * 50)
print("시리즈: DEXKOUS (USD/KRW 환율)")

# FRED에서 환율 데이터 수집
df_fx = web.DataReader('DEXKOUS', 'fred', start_date, end_date)
df_fx.columns = ['환율']

# 결측치 처리 (주말/공휴일 → 전일 값으로 채움)
df_fx = df_fx.ffill()

df_fx.tail()

[환율 데이터 수집 - FRED]
시리즈: DEXKOUS (USD/KRW 환율)


,환율
DATE,
2025-12-29,1433.88
2025-12-30,1439.42
2025-12-31,1444.55
2026-01-01,1444.55
2026-01-02,1444.45


In [14]:
# 주가 데이터와 환율 데이터 병합
df = df.merge(df_fx, left_index=True, right_index=True, how='left')
df.head()

,Open,High,Low,Close,Volume,Change,환율
Date,,,,,,,
2025-01-10,56100,56500,55200,55300,16059223,-0.014260,1473.36
2025-01-13,54600,55000,54100,54100,16868600,-0.021700,1466.87
2025-01-14,54200,54600,53700,53900,17465926,-0.003697,1459.50
2025-01-15,54100,54700,53500,53700,18625025,-0.003711,1456.50
2025-01-16,54200,55000,54100,54300,18627298,0.011173,1456.16


In [15]:
# 환율 결측치 처리 (주식 거래일과 환율 데이터 불일치 시)
df['환율'] = df['환율'].ffill().bfill()  # 앞의 값으로 먼저 채우고, 그래도 남으면 뒤의 값으로 채운다

---
## 3. 특성 공학 (Feature Engineering)

주가 예측에 유용한 특성들을 생성합니다. (20차시 내용 + 환율 특성 추가)

In [16]:
# 특성 생성 (핵심 특성만 간단히 - 20차시와 동일)
df['전일종가'] = df['Close'].shift(1)           # 전일 종가
df['수익률'] = df['Close'].pct_change() * 100   # 전일 대비 수익률 (%)
df['5일이동평균'] = df['Close'].rolling(5).mean()  # 5일 이동평균
df['거래량비율'] = df['Volume'] / df['Volume'].rolling(20).mean()  # 평균 대비 거래량

# 환율 특성 (FRED 데이터 활용)
df['환율변화율'] = df['환율'].pct_change() * 100  # 전일 대비 환율 변화율 (%)

# 결측치 제거
df = df.dropna()
df.head()

,Open,High,Low,Close,Volume,Change,환율,전일종가,수익률,5일이동평균,거래량비율,환율변화율
Date,,,,,,,,,,,,
2025-02-12,55100,55900,54500,55800,26428596,0.001795,1453.49,55700.0,0.179533,54960.0,1.305524,0.081250
2025-02-13,56100,56400,55600,55800,22448377,0.000000,1446.95,55800.0,0.000000,55320.0,1.091681,-0.449951
2025-02-14,56000,57300,56000,56000,23979779,0.003584,1439.89,55800.0,0.358423,55780.0,1.146333,-0.487923
2025-02-17,56100,56500,55700,56000,11916027,0.000000,1439.89,56000.0,0.000000,55860.0,0.577293,0.000000
2025-02-18,56200,57200,55900,56900,22131007,0.016071,1438.83,56000.0,1.607143,56100.0,1.063148,-0.073617


In [17]:
# 타겟 변수 생성: 다음날 상승(1) / 하락(0)
df['다음날종가'] = df['Close'].shift(-1)
df['다음날상승'] = (df['다음날종가'] > df['Close']).astype(int)
df = df.dropna()
df.head()

,Open,High,Low,Close,Volume,Change,환율,전일종가,수익률,5일이동평균,거래량비율,환율변화율,다음날종가,다음날상승
Date,,,,,,,,,,,,,,
2025-02-12,55100,55900,54500,55800,26428596,0.001795,1453.49,55700.0,0.179533,54960.0,1.305524,0.081250,55800.0,0
2025-02-13,56100,56400,55600,55800,22448377,0.000000,1446.95,55800.0,0.000000,55320.0,1.091681,-0.449951,56000.0,1
2025-02-14,56000,57300,56000,56000,23979779,0.003584,1439.89,55800.0,0.358423,55780.0,1.146333,-0.487923,56000.0,0
2025-02-17,56100,56500,55700,56000,11916027,0.000000,1439.89,56000.0,0.000000,55860.0,0.577293,0.000000,56900.0,1
2025-02-18,56200,57200,55900,56900,22131007,0.016071,1438.83,56000.0,1.607143,56100.0,1.063148,-0.073617,58700.0,1


---
## 4. LogisticRegression 모델 학습

### 예측 대상 정의
**타겟(y)**: 다음날 상승(1) 또는 하락(0)

In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 특성과 타겟 분리 (환율 특성 포함)
feature_cols = ['전일종가', '수익률', '5일이동평균', '거래량비율', '환율변화율']

X = df[feature_cols]
y = df['다음날상승']
X.shape, y.shape

((222, 5), (222,))

In [22]:
# 데이터 분할 (시계열이므로 섞지 않음 - 80:20)
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((177, 5), (45, 5), (177,), (45,))

In [23]:
# 스케일링 (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [25]:
# LogisticRegression 모델 학습
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [26]:
# 예측
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]  # 상승 확률

---
## 5. 모델 평가 및 시각화

### 분류 평가 지표
| 지표 | 수식 | 의미 |
|------|------|------|
| Accuracy | (TP+TN)/전체 | 전체 정확도 |

In [27]:
from sklearn.metrics import accuracy_score

# 평가 지표 계산
accuracy = accuracy_score(y_test, y_pred)

print("[모델 평가 결과]")
print("=" * 50)
print(f"Accuracy:  {accuracy:.1%} (전체 중 맞춘 비율)")

[모델 평가 결과]
Accuracy:  57.8% (전체 중 맞춘 비율)


---
## 학습 정리

### 1. 이진분류 워크플로우
```
데이터 수집 (주가+환율) → 타겟 생성(0/1) → 특성 생성 → 스케일링 → 모델 학습 → 평가
```

### 2. 핵심 코드
```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas_datareader.data as web

# 환율 데이터 수집 (FRED)
df_fx = web.DataReader('DEXKOUS', 'fred', start_date, end_date)

# 타겟 생성: 내일 주가가 오를지 내릴지
df['다음날상승'] = (df['종가'].shift(-1) > df['종가']).astype(int)

# 모델 학습 (주가 + 환율 특성)
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# 예측 및 평가
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
```

### 3. 주요 평가 지표
| 지표 | 용도 |
|------|------|
| Accuracy | 전체 정확도 (균형 데이터) |

### 4. 환율 특성 활용
- FRED API로 USD/KRW 환율 수집 (Module 02 연계)
- 환율변화율을 특성으로 추가
- 수출 기업(삼성전자)은 환율과 상관관계 분석 가능

### 5. 주의사항
- 주가 방향 예측도 매우 어려운 문제
- 50% 이상이면 랜덤보다 나음
- 과적합 및 데이터 누수 주의

---

### 다음 차시 예고
- 22차시: 모델 성능 평가 심화
  - 회귀 vs 분류 평가 비교
  - ROC Curve, AUC
  - 교차 검증